In [ ]:
!pip install databricks-sdk -U
dbutils.library.restartPython()

In [ ]:
import json
import requests
from databricks.sdk import WorkspaceClient

# ========================================
# Widget Parameters (for Job/CI-CD integration)
# ========================================
dbutils.widgets.text("space_id", "", "Space ID to export (required)")
dbutils.widgets.text("output_file", "../genie_definition/genie_space.json", "Output JSON file path")

# ========================================
# Get parameter values
# ========================================
SPACE_ID = dbutils.widgets.get("space_id")
OUTPUT_FILE = dbutils.widgets.get("output_file")

# Validate required parameters
if not SPACE_ID:
    raise ValueError("space_id parameter is required. Please provide the Genie Space ID to export.")

print(f"🚀 Exporting Genie Space: {SPACE_ID}")

# ========================================
# Initialize client and get auth headers
# ========================================
w = WorkspaceClient()
workspace_url = w.config.host

# Use SDK's built-in auth (works on serverless, clusters, and all auth types)
headers = w.config.authenticate()
headers["Content-Type"] = "application/json"

# ========================================
# Call Get Space API
# ========================================
response = requests.get(
    f"{workspace_url}/api/2.0/genie/spaces/{SPACE_ID}",
    headers=headers,
    params={"include_serialized_space": "true"}
)

response.raise_for_status()
space_definition = response.json()

print(f"\n✅ Successfully retrieved Genie space:")
print(f"   - Title: {space_definition.get('title', 'N/A')}")
print(f"   - Space ID: {space_definition.get('space_id', SPACE_ID)}")

# ========================================
# Extract and save the serialized space
# ========================================
# Parse the serialized space JSON string
serialized_space = json.loads(space_definition["serialized_space"])
    
# Save to file with pretty formatting
with open(OUTPUT_FILE, 'w') as f:
    json.dump(serialized_space, f, indent=2)
    
print(f"\n💾 Exported to: {OUTPUT_FILE}")

# ========================================
# Output for downstream tasks
# ========================================
print("\n🎉 Export complete!")

# Return output for job chaining
dbutils.notebook.exit(json.dumps({
    "status": "exported",
    "space_id": SPACE_ID,
    "title": space_definition.get('title', 'N/A'),
    "output_file": OUTPUT_FILE
}))